In [1]:
!pip install yfinance prophet pandas numpy matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### **1. Importing Packages and Libraries**

In [3]:
# Importing all the necessary libraries and packages

import yfinance as yf
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from datetime import timedelta


### **2. Downloading Data**

Downloading the stock prices for the top 25 companies worldwide for the past 6 years.

In [4]:
# Creating a list of all the companies tickers
tickers = [
    "AAPL","MSFT","NVDA","AMZN","GOOG",
    "META","TSLA","BRK-B","LLY","AVGO",
    "ASML","TSM","JPM","V","MA",
    "WMT","XOM","JNJ","NVO","PG",
    "COST","HD","KO","PEP","TM"
]

# Downloading Data for each ticker
def download_stock_data(ticker_list):
    stock_data = {}

    for ticker in ticker_list:
        # Ensuring all tickers' data is downloaded
        print(f"Downloading {ticker}")

        df = yf.download(ticker, period="6y")
        df = df.reset_index()
        df = df[["Date","Close"]]
        df.columns = ["ds","y"]

        stock_data[ticker] = df

    return stock_data


In [11]:
stock_data = download_stock_data(tickers)
print(stock_data)

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


...

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed

{'AAPL':              ds           y
0    2020-03-16   58.524204
1    2020-03-17   61.097538
2    2020-03-18   59.601856
3    2020-03-19   59.145195
4    2020-03-20   55.390327
...         ...         ...
1502 2026-03-09  259.880005
1503 2026-03-10  260.829987
1504 2026-03-11  260.809998
1505 2026-03-12  255.759995
1506 2026-03-13  250.119995

[1507 rows x 2 columns], 'MSFT':              ds           y
0    2020-03-16  128.636322
1    2020-03-17  139.227829
2    2020-03-18  133.366898
3    2020-03-19  135.561203
4    2020-03-20  130.469666
...         ...         ...
1502 2026-03-09  409.410004
1503 2026-03-10  405.760010
1504 2026-03-11  404.880005
1505 2026-03-12  401.859985
1506 2026-03-13  395.549988

[1507 rows x 2 columns], 'NVDA':              ds           y
0    2020-03-16    4.890823
1    2020-03-17    5.410536
2    2020-03-18    5.050697
3    2020-03-19    5.303456
4    2020-03-20    5.123662
...         ...         ...
1502 2026-03-09  182.640106
1503 2026-03-10  184.760010

### **3. Feature Engineeting**

Generating new features to optimize the model's predictions. 

**1. "return":** Calculating the percentage change in price from one day to the next from stock prices. This will help the model detect positive and negative patarns and sudden or drastic changes.

**2. "volatility":** Calculating the standard deviation of returns over the past 7 days. This will allow the model to learn large movements and trend continuation relationships.

**3. "m_7":** Calculating the weekly rolling average to smooth out daily noise

**4. "m_30":** Calculating the weekly and monthly rolling averages to capture long-term trends


In [5]:
def engineer_features(stock_data):
    feature_data = {}

    for ticker, df in stock_data.items():
        df["return"] = df["y"].pct_change()
        df["volatility"] = df["return"].rolling(7).std()
        df["ma_7"] = df["y"].rolling(7).mean()
        df["ma_30"] = df["y"].rolling(30).mean()

        df = df.dropna()

        feature_data[ticker] = df

    return feature_data


In [14]:
feature_data = engineer_features(stock_data)

### **4. Metrics Function**

In [6]:
# Building a function to evaluate the forecasting model

def evaluate_model(actual, predicted):
    r2 = r2_score(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    actual_direction = np.sign(np.diff(actual))
    pred_direction = np.sign(np.diff(predicted))
    directional_accuracy = (actual_direction == pred_direction).mean()

    return r2, rmse, directional_accuracy


### **5. Forecasting Stock Prices**

Building and training a Prophet model to forecast future stock prices for each of the companies in the tickers list. 

The function will train the model recursively, where it will predict the first three months of each yer based on the years before it.

In [7]:
def train_prophet_models(feature_data):

    results = []
    predictions_store = {}

    for ticker, df in feature_data.items():

        print(f"Training model for {ticker}")

        yearly_forecasts = []

        start_year = df["ds"].dt.year.min() + 1
        end_year = df["ds"].dt.year.max()

        for year in range(start_year, end_year):

            train = df[df["ds"].dt.year < year]

            test = df[
                (df["ds"].dt.year == year) &
                (df["ds"].dt.month <= 3)
            ]

            if len(test) == 0:
                continue

            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=True
            )

            model.fit(train[["ds","y"]])

            future_dates = pd.date_range(
                start=f"{year}-01-01",
                end=f"{year}-03-31"
            )

            future = pd.DataFrame({"ds": future_dates})

            forecast = model.predict(future)

            predicted = forecast["yhat"].values
            actual = test["y"].values[:len(predicted)]

            r2, rmse, dir_acc = evaluate_model(
                actual,
                predicted[:len(actual)]
            )

            results.append({
                "ticker": ticker,
                "year": year,
                "R2": r2,
                "RMSE": rmse,
                "Directional_Accuracy": dir_acc
            })

            yearly_forecasts.append((year,test,forecast))

        predictions_store[ticker] = yearly_forecasts

    return results, predictions_store


In [ ]:
results, predictions_store = train_prophet_models(feature_data)
results_df = pd.DataFrame(results)
# results_df.to_csv("evaluation_results.csv", index=False)

Training model for AAPL


12:50:28 - cmdstanpy - INFO - Chain [1] start processing
12:50:29 - cmdstanpy - INFO - Chain [1] done processing
12:50:30 - cmdstanpy - INFO - Chain [1] start processing
12:50:30 - cmdstanpy - INFO - Chain [1] done processing
12:50:31 - cmdstanpy - INFO - Chain [1] start processing
12:50:31 - cmdstanpy - INFO - Chain [1] done processing
12:50:32 - cmdstanpy - INFO - Chain [1] start processing
12:50:33 - cmdstanpy - INFO - Chain [1] done processing
12:50:34 - cmdstanpy - INFO - Chain [1] start processing
12:50:36 - cmdstanpy - INFO - Chain [1] done processing


Training model for MSFT


12:50:36 - cmdstanpy - INFO - Chain [1] start processing
12:50:36 - cmdstanpy - INFO - Chain [1] done processing
12:50:37 - cmdstanpy - INFO - Chain [1] start processing
12:50:37 - cmdstanpy - INFO - Chain [1] done processing
12:50:38 - cmdstanpy - INFO - Chain [1] start processing
12:50:38 - cmdstanpy - INFO - Chain [1] done processing
12:50:39 - cmdstanpy - INFO - Chain [1] start processing
12:50:40 - cmdstanpy - INFO - Chain [1] done processing
12:50:41 - cmdstanpy - INFO - Chain [1] start processing
12:50:42 - cmdstanpy - INFO - Chain [1] done processing


Training model for NVDA


...

12:52:38 - cmdstanpy - INFO - Chain [1] start processing
12:52:38 - cmdstanpy - INFO - Chain [1] done processing
12:52:39 - cmdstanpy - INFO - Chain [1] start processing
12:52:39 - cmdstanpy - INFO - Chain [1] done processing
12:52:40 - cmdstanpy - INFO - Chain [1] start processing
12:52:41 - cmdstanpy - INFO - Chain [1] done processing
12:52:42 - cmdstanpy - INFO - Chain [1] start processing
12:52:43 - cmdstanpy - INFO - Chain [1] done processing
12:52:44 - cmdstanpy - INFO - Chain [1] start processing
12:52:45 - cmdstanpy - INFO - Chain [1] done processing


Training model for PEP


12:52:45 - cmdstanpy - INFO - Chain [1] start processing
12:52:46 - cmdstanpy - INFO - Chain [1] done processing
12:52:46 - cmdstanpy - INFO - Chain [1] start processing
12:52:46 - cmdstanpy - INFO - Chain [1] done processing
12:52:47 - cmdstanpy - INFO - Chain [1] start processing
12:52:47 - cmdstanpy - INFO - Chain [1] done processing
12:52:48 - cmdstanpy - INFO - Chain [1] start processing
12:52:49 - cmdstanpy - INFO - Chain [1] done processing
12:52:49 - cmdstanpy - INFO - Chain [1] start processing
12:52:50 - cmdstanpy - INFO - Chain [1] done processing


Training model for TM


12:52:51 - cmdstanpy - INFO - Chain [1] start processing
12:52:51 - cmdstanpy - INFO - Chain [1] done processing
12:52:51 - cmdstanpy - INFO - Chain [1] start processing
12:52:52 - cmdstanpy - INFO - Chain [1] done processing
12:52:52 - cmdstanpy - INFO - Chain [1] start processing
12:52:53 - cmdstanpy - INFO - Chain [1] done processing
12:52:53 - cmdstanpy - INFO - Chain [1] start processing
12:52:54 - cmdstanpy - INFO - Chain [1] done processing
12:52:55 - cmdstanpy - INFO - Chain [1] start processing
12:52:56 - cmdstanpy - INFO - Chain [1] done processing


### **6. Visualizing Forecasting Results**

In [8]:
def create_visualizations(feature_data, predictions_store):

    for ticker, df in feature_data.items():
        forecasts = predictions_store[ticker]

        # Yearly Plots
        for year, actual, forecast in forecasts:
            plt.figure(figsize=(10,5))
            plt.plot(actual["ds"], actual["y"], label="Actual")
            plt.plot(forecast["ds"], forecast["yhat"], label="Prediction")
            plt.title(f"{ticker} Forecast Jan–Mar {year}")
            plt.legend()
            plt.savefig(
                f"plots/yearly/{ticker}_{year}.png"
            )
            plt.close()


        # Last 6 years + last 3 months forecast
        model = Prophet()
        model.fit(df[["ds","y"]])
        last_3m = pd.date_range(
            start=df["ds"].max() - timedelta(days=90),
            end=df["ds"].max()
        )
        future_last = pd.DataFrame({"ds": last_3m})
        forecast_last = model.predict(future_last)
        
        plt.figure(figsize=(12,6))
        plt.plot(df["ds"], df["y"], label="Actual")
        plt.plot(
            forecast_last["ds"],
            forecast_last["yhat"],
            label="Past 3M Prediction"
        )
        plt.title(f"{ticker} Past 6 Years + Prediction")
        plt.legend()
        plt.savefig(
            f"plots/historical/{ticker}_history.png"
        )
        plt.close()


        # Next 3 months forecast
        future = model.make_future_dataframe(periods=90)
        forecast_future = model.predict(future)
        plt.figure(figsize=(12,6))
        plt.plot(df["ds"], df["y"], label="Actual")
        plt.plot(
            forecast_future["ds"],
            forecast_future["yhat"],
            label="Next 3M Forecast"
        )
        plt.title(f"{ticker} Future Forecast")
        plt.legend()
        plt.savefig(
            f"plots/future/{ticker}_future.png"
        )
        plt.close()


In [ ]:
create_visualizations(feature_data, predictions_store)